# LeetCode #172 — Factorial Trailing Zeroes: đếm thừa số 5

**Mức độ:** Medium · **Chủ đề:** Math

Skeleton trên LeetCode:

```python
class Solution:
    def trailingZeroes(self, n: int) -> int:
```

Mục tiêu của notebook:

1. Mỗi chữ số 0 ở cuối = một thừa số `10 = 2 × 5`.
2. Trong `n!` thừa số 2 luôn **nhiều hơn** thừa số 5 → chỉ cần **đếm số thừa số 5**.
3. **Công thức Legendre**: `⌊n/5⌋ + ⌊n/25⌋ + ⌊n/125⌋ + …` — chạy `O(log n)`.
4. Mở rộng: số 0 cuối trong cơ số bất kỳ, bài toán ngược (#793).

# 1. Bài toán là gì?

Cho số nguyên `n`, trả về số **chữ số 0 ở cuối** của `n!`.

Nhắc lại: `n! = n × (n − 1) × … × 3 × 2 × 1` (và `0! = 1`).

**Constraints (theo đề):**

- `0 <= n <= 10^4`

**Follow up (theo đề):** bạn có thể viết lời giải chạy trong thời gian **logarit** không?

---

## Ví dụ

```text
n = 3   ->  0     (3! = 6)
n = 5   ->  1     (5! = 120)
n = 0   ->  0     (0! = 1)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Số học cơ bản — số nguyên tố, sàng, ƯCLN, lũy thừa của p trong n!

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Số học cơ bản — số nguyên tố, sàng, ƯCLN, lũy thừa của p trong n!

### Số nguyên tố và kiểm tra bằng thử chia

`p >= 2` là **số nguyên tố** nếu chỉ chia hết cho 1 và chính nó. Để kiểm tra `n`, chỉ cần thử các ước `d` với `d² <= n`: nếu `n = a · b` với `a <= b` thì `a <= √n`.

In [1]:
from math import isqrt, gcd

def is_prime_demo(n):
    if n < 2:
        return False
    for d in range(2, isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

print([x for x in range(30) if is_prime_demo(x)])
print("isqrt(10**12 + 1) =", isqrt(10**12 + 1), "(căn nguyên chính xác, không lỗi làm tròn số thực)")

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
isqrt(10**12 + 1) = 1000000 (căn nguyên chính xác, không lỗi làm tròn số thực)


### Sàng Eratosthenes: mọi số nguyên tố `< n` trong O(n log log n)

Ý tưởng: đi từ 2 lên; gặp số `p` chưa bị gạch thì `p` nguyên tố, **gạch mọi bội** của nó. Bắt đầu gạch từ `p²` (các bội nhỏ hơn đã bị gạch bởi ước nhỏ hơn), và chỉ cần `p² < n`.

```text
2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
    x   x   x x  x     x     x  x  x     x       (gạch bội của 2 từ 4, của 3 từ 9)
```

In [2]:
def sieve_demo(n):
    if n < 3:
        return []
    is_p = bytearray([1]) * n
    is_p[0] = is_p[1] = 0
    for p in range(2, isqrt(n - 1) + 1):
        if is_p[p]:
            is_p[p * p::p] = bytes(len(range(p * p, n, p)))
    return [i for i in range(n) if is_p[i]]

print(sieve_demo(30), len(sieve_demo(10**5)))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29] 9592


### Ước chung lớn nhất: thuật toán Euclid

`gcd(a, b) = gcd(b, a mod b)`, dừng khi `b = 0`. Số bước `O(log min(a, b))`. Python có sẵn `math.gcd`.

In [3]:
def gcd_demo(a, b):
    steps = []
    while b:
        steps.append((a, b))
        a, b = b, a % b
    return a, steps

g, st = gcd_demo(252, 105)
print("gcd(252, 105) =", g, "| các bước:", st, "| math.gcd:", gcd(252, 105))

gcd(252, 105) = 21 | các bước: [(252, 105), (105, 42), (42, 21)] | math.gcd: 21


Ứng dụng: các bước nhảy `+k (mod n)` xuất phát từ 0 đi qua đúng `n / gcd(n, k)` vị trí rồi quay lại 0 — chia các vị trí thành `gcd(n, k)` vòng.

In [4]:
n_demo, k_demo = 6, 4
start, cycle = 0, [0]
while (cycle[-1] + k_demo) % n_demo != start:
    cycle.append((cycle[-1] + k_demo) % n_demo)
print(f"n={n_demo}, k={k_demo}: vòng từ 0 = {cycle}, độ dài {len(cycle)} = n / gcd = {n_demo // gcd(n_demo, k_demo)}")

n=6, k=4: vòng từ 0 = [0, 4, 2], độ dài 3 = n / gcd = 3


### Lũy thừa của số nguyên tố p trong n! (công thức Legendre)

Số lần `p` xuất hiện trong phân tích `n!` là `⌊n/p⌋ + ⌊n/p²⌋ + ⌊n/p³⌋ + …`: `⌊n/p⌋` số chia hết cho `p` mỗi số góp 1, `⌊n/p²⌋` số chia hết cho `p²` góp thêm 1, …

In [5]:
from math import factorial

def legendre_demo(n, p):
    total, q = 0, p
    while q <= n:
        total += n // q
        q *= p
    return total

def count_p_brute(n, p):
    f, c = factorial(n), 0
    while f % p == 0:
        f //= p; c += 1
    return c

for n_, p_ in [(10, 2), (25, 5), (100, 3)]:
    print(f"lũy thừa của {p_} trong {n_}! = {legendre_demo(n_, p_)} (đếm trực tiếp: {count_p_brute(n_, p_)})")

lũy thừa của 2 trong 10! = 8 (đếm trực tiếp: 8)
lũy thừa của 5 trong 25! = 6 (đếm trực tiếp: 6)
lũy thừa của 3 trong 100! = 48 (đếm trực tiếp: 48)


### Lỗi hay gặp

- Kiểm tra nguyên tố tới `n` thay vì `√n` → chậm; dùng `int(n ** 0.5)` có thể lệch với số lớn → dùng `isqrt`.
- Sàng gạch từ `2p` thay vì `p²` (vẫn đúng nhưng chậm hơn); quên 0 và 1 không nguyên tố.
- Tính `n!` thật rồi đếm → số khổng lồ, rất chậm.

## 2.2 Áp dụng vào bài này

Mỗi số 0 ở cuối là một thừa số `10 = 2 · 5`. Trong `n!` thừa số 2 luôn nhiều hơn thừa số 5, nên số chữ số 0 bằng **số mũ của 5 trong `n!`** — chính là **công thức Legendre** `⌊n/5⌋ + ⌊n/25⌋ + ⌊n/125⌋ + …`, `O(log n)` mà không cần tính `n!`.

# 3. Nền tảng

In [6]:
from math import factorial
import random, time, sys

# Python 3.11+ chặn đổi số nguyên > 4300 chữ số sang chuỗi (chống tấn công DoS).
# Ta cần str(10000!) ở mục 4 nên tắt giới hạn này.
sys.set_int_max_str_digits(0)

for n in [3, 5, 10, 15, 25]:
    f = factorial(n)
    s = str(f)
    print(f"{n:>2}! = {f:<30} số 0 cuối: {len(s) - len(s.rstrip('0'))}")

 3! = 6                              số 0 cuối: 0
 5! = 120                            số 0 cuối: 1
10! = 3628800                        số 0 cuối: 2
15! = 1307674368000                  số 0 cuối: 3
25! = 15511210043330985984000000     số 0 cuối: 6


## Vì sao chỉ đếm thừa số 5?

Một số kết thúc bằng `k` chữ số 0 ⟺ nó chia hết cho `10^k = 2^k · 5^k`. Vậy số chữ số 0 cuối = `min(số thừa số 2, số thừa số 5)` trong phân tích thừa số nguyên tố.

Trong `1, 2, …, n`: cứ 2 số có một số chẵn, nhưng 5 số mới có một bội của 5 → thừa số 2 **luôn nhiều hơn** → đáp án = **số thừa số 5**.

In [7]:
def count_factor(n_fact_upto, p):
    """Đếm số mũ của p trong n! bằng cách phân tích từng số 1..n (chậm, để minh họa)."""
    total = 0
    for k in range(1, n_fact_upto + 1):
        while k % p == 0:
            total += 1
            k //= p
    return total

for n in [10, 25, 100]:
    print(f"{n}!: thừa số 2 = {count_factor(n, 2):>3}, thừa số 5 = {count_factor(n, 5):>3}")

10!: thừa số 2 =   8, thừa số 5 =   2
25!: thừa số 2 =  22, thừa số 5 =   6
100!: thừa số 2 =  97, thừa số 5 =  24


# 4. Baseline: tính `n!` rồi đếm số 0

Python có số nguyên lớn nên **làm được** (và khá nhanh, xem số đo bên dưới), nhưng `10000!` có hơn 35 nghìn chữ số — tốn bộ nhớ và thời gian hơn hẳn vài phép chia của lời giải chính (và từ Python 3.11 phải gọi `sys.set_int_max_str_digits(0)` như ở mục 3, nếu không `str(...)` báo `ValueError` khi số vượt 4300 chữ số). Trong ngôn ngữ khác, `n!` tràn số ngay từ `n = 21`.

In [8]:
def zeros_by_factorial(n: int) -> int:
    s = str(factorial(n))
    return len(s) - len(s.rstrip("0"))

t0 = time.time(); z = zeros_by_factorial(10**4); t1 = time.time()
print(f"10^4!: {len(str(factorial(10**4)))} chữ số, {z} số 0 cuối, mất {t1 - t0:.3f}s")

10^4!: 35660 chữ số, 2499 số 0 cuối, mất 0.011s


# 5. Ý tưởng chính: công thức Legendre

Đếm thừa số 5 trong `1 × 2 × … × n`:

- Mỗi bội của `5` góp **ít nhất một** thừa số 5: có `⌊n/5⌋` số như vậy.
- Mỗi bội của `25 = 5²` góp **thêm một** (thừa số thứ hai): `⌊n/25⌋` số.
- Mỗi bội của `125 = 5³` góp thêm một nữa: `⌊n/125⌋`, …

```text
đáp án = ⌊n/5⌋ + ⌊n/25⌋ + ⌊n/125⌋ + …       (dừng khi 5^k > n)

n = 100: 100//5 = 20, 100//25 = 4, 100//125 = 0  → 24
```

Cách viết gọn: lặp `n //= 5` và cộng dồn — vì `⌊⌊n/5⌋/5⌋ = ⌊n/25⌋`.

In [9]:
def trailing_zeroes(n: int) -> int:
    count = 0
    while n >= 5:
        n //= 5
        count += n
    return count

for n in [3, 5, 0, 25, 100, 10**4]:
    print(f"n = {n:>5}: {trailing_zeroes(n)}")

n =     3: 0
n =     5: 1
n =     0: 0
n =    25: 6
n =   100: 24
n = 10000: 2499


# 6. Mô phỏng

In [10]:
def trace_zeroes(n):
    count, p = 0, 5
    while p <= n:
        print(f"   bội của {p:>5}: {n} // {p} = {n // p:>4} → cộng thêm {n // p}")
        count += n // p
        p *= 5
    print(f"→ {count} số 0 cuối")
    return count

_ = trace_zeroes(1000)

   bội của     5: 1000 // 5 =  200 → cộng thêm 200
   bội của    25: 1000 // 25 =   40 → cộng thêm 40
   bội của   125: 1000 // 125 =    8 → cộng thêm 8
   bội của   625: 1000 // 625 =    1 → cộng thêm 1
→ 249 số 0 cuối


# 7. Phiên bản đúng format LeetCode

In [11]:
class Solution:
    def trailingZeroes(self, n: int) -> int:
        count = 0
        while n >= 5:
            n //= 5
            count += n
        return count

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<10} PASS  {actual}")


sol = Solution()
check("n = 3", sol.trailingZeroes(3), 0)
check("n = 5", sol.trailingZeroes(5), 1)
check("n = 0", sol.trailingZeroes(0), 0)
check("n = 25", sol.trailingZeroes(25), 6)
check("n = 10^4", sol.trailingZeroes(10**4), 2499)

for n in range(0, 1500):
    assert sol.trailingZeroes(n) == zeros_by_factorial(n), n
print("n = 0..1499: khớp đếm trực tiếp trên n!")

t0 = time.perf_counter()
for _ in range(10**5):
    sol.trailingZeroes(10**4)
t1 = time.perf_counter()
print(f"10^5 lần gọi với n = 10^4: {t1 - t0:.3f}s (mỗi lần chỉ 5 vòng lặp: 10000 → 2000 → 400 → 80 → 16 → 3)")

n = 3      PASS  0
n = 5      PASS  1
n = 0      PASS  0
n = 25     PASS  6
n = 10^4   PASS  2499


n = 0..1499: khớp đếm trực tiếp trên n!
10^5 lần gọi với n = 10^4: 0.101s (mỗi lần chỉ 5 vòng lặp: 10000 → 2000 → 400 → 80 → 16 → 3)


# 8. Vì sao đúng? Độ phức tạp?

Số mũ của 5 trong `n!` bằng `Σ_{k=1..n} v5(k)` với `v5(k)` = số mũ của 5 trong `k`. Viết `v5(k) = Σ_{j>=1} [5^j | k]` (số lũy thừa `5^j` chia hết `k`). Đổi thứ tự tổng:

```text
Σ_k Σ_j [5^j | k] = Σ_j (số k ≤ n chia hết cho 5^j) = Σ_j ⌊n / 5^j⌋
```

Đó là công thức Legendre (đúng cho mọi số nguyên tố `p`). Kết hợp "thừa số 2 nhiều hơn" → đáp án.

```text
Time:  O(log₅ n)
Space: O(1)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Chỉ đếm `n // 5`

In [13]:
def wrong_only_5(n):
    return n // 5

for n in [24, 25, 125]:
    print(f"n = {n}: sai {wrong_only_5(n)} | đúng {sol.trailingZeroes(n)}")

n = 24: sai 4 | đúng 4
n = 25: sai 5 | đúng 6
n = 125: sai 25 | đúng 31


`25 = 5 × 5` góp **hai** thừa số 5; `125` góp ba. Bỏ qua các lũy thừa cao hơn sẽ đếm thiếu.

## Lỗi 2: Đếm số 0 của từng thừa số

In [14]:
def wrong_count_zero_digits(n):
    return sum(str(k).count("0") for k in range(1, n + 1))   # SAI: 4 × 5 = 20 không có số 0 nào trong 4 hay 5

for n in [5, 10, 25]:
    print(f"n = {n}: sai {wrong_count_zero_digits(n)} | đúng {sol.trailingZeroes(n)}")

n = 5: sai 0 | đúng 1
n = 10: sai 1 | đúng 2
n = 25: sai 2 | đúng 6


## Lỗi 3: Tính `n!` trong ngôn ngữ có giới hạn số nguyên

Với số 64 bit, `21! ≈ 5.1·10^19` đã vượt `2^63 − 1 ≈ 9.2·10^18`. Python không tràn nhưng tốn thời gian và bộ nhớ (mục 4). Công thức Legendre không cần tính `n!`.

In [15]:
print("21! =", factorial(21), "> 2^63 - 1 =", 2**63 - 1, "?", factorial(21) > 2**63 - 1)

21! = 51090942171709440000 > 2^63 - 1 = 9223372036854775807 ? True


## Lỗi 4: Vòng lặp nhân `p *= 5` trong ngôn ngữ có tràn số

Viết `while p <= n: … p *= 5` có thể tràn khi `n` gần giới hạn kiểu dữ liệu (`p` vượt `n` nhưng tràn thành số âm). Cách `n //= 5` an toàn hơn vì chỉ làm số **nhỏ đi**.

# 10. Mẹo nhận diện pattern

- **Số 0 cuối / chia hết cho `b^k`** → phân tích thừa số nguyên tố, đếm số mũ, lấy `min`.
- **Số mũ của số nguyên tố `p` trong `n!`** → Legendre `Σ ⌊n/p^j⌋`.
- **Hàm đếm đơn điệu + câu hỏi "bao nhiêu `x` cho kết quả `K`"** → tìm nhị phân trên `x` (bài tập 5).

Bài liên quan: **#793 Preimage Size of Factorial Zeroes Function** (Hard), **#204 Count Primes**, **#2117 Abbreviating the Product of a Range** (Hard), **#1015**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tính số 0 cuối của `2026!`.

## Bài 2 — Điền chỗ trống

In [16]:
def exercise_2_trailing_zeroes(n: int) -> int:
    count = 0
    # TODO: lặp chia n cho 5 và cộng dồn
    pass
    return count


# assert exercise_2_trailing_zeroes(3) == 0
# assert exercise_2_trailing_zeroes(5) == 1
# assert exercise_2_trailing_zeroes(25) == 6
# assert exercise_2_trailing_zeroes(10000) == 2499

## Bài 3 — Biến thể: số 0 cuối của **tích một mảng**

Cho mảng số nguyên dương, đếm số 0 ở cuối của tích tất cả phần tử, **không** tính tích. Ở đây thừa số 2 **không** chắc nhiều hơn.

In [17]:
def exercise_3_zeros_of_product(nums: list) -> int:
    # TODO
    pass


# assert exercise_3_zeros_of_product([4, 25, 7]) == 2
# assert exercise_3_zeros_of_product([5, 5, 5, 2]) == 1
# assert exercise_3_zeros_of_product([3, 7]) == 0

## Bài 4 — Biến thể: số 0 cuối của `n!` trong **cơ số** `b`

Phân tích `b = ∏ p_i^{e_i}`. Số 0 cuối trong cơ số `b` = `min_i ⌊ v_{p_i}(n!) / e_i ⌋`, với `v_p(n!)` tính bằng Legendre.

In [18]:
def exercise_4_trailing_zeroes_base(n: int, b: int) -> int:
    # TODO
    pass


# assert exercise_4_trailing_zeroes_base(10, 10) == 2
# assert exercise_4_trailing_zeroes_base(5, 2) == 3
# assert exercise_4_trailing_zeroes_base(10, 12) == 4
# assert exercise_4_trailing_zeroes_base(1, 7) == 0

## Bài 5 — Mini challenge: Preimage Size of Factorial Zeroes Function (LeetCode #793, Hard)

Có bao nhiêu số nguyên `x >= 0` sao cho `n = x` thì `x!` có đúng `k` số 0 cuối? (`0 <= k <= 10^9`.)

Nhận xét: `f(x)` = số 0 cuối của `x!` là hàm **không giảm** theo `x`, tăng đúng tại các bội của 5. Đáp án luôn là `0` hoặc `5`.

In [19]:
def exercise_5_preimage_size(k: int) -> int:
    # TODO
    pass


# assert exercise_5_preimage_size(0) == 5
# assert exercise_5_preimage_size(5) == 0
# assert exercise_5_preimage_size(3) == 5
# assert exercise_5_preimage_size(10**9) == 5

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_zeroes(2026)` sau khi tự làm.

**Bài 3**
- Mức 1: đếm tổng số mũ của 2 và của 5 trong từng phần tử, đáp án là `min`.

**Bài 4**
- Mức 1: phân tích `b` bằng chia thử `p = 2, 3, …` tới `p*p <= b`.
- Mức 2: Legendre tổng quát: `v = 0; m = n; while m: m //= p; v += m`.

**Bài 5**
- Mức 1: tìm nhị phân `x` nhỏ nhất với `f(x) >= k` (cận trên `5k` là đủ vì `f(5k) >= k`).
- Mức 2: nếu `f(x) == k` thì `x, x+1, …, x+4` đều cho `k` (giá trị chỉ đổi ở bội của 5) → `5`; ngược lại `0`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [20]:
_ = trace_zeroes(2026)
print("kiểm tra trên 2026!:", zeros_by_factorial(2026))

   bội của     5: 2026 // 5 =  405 → cộng thêm 405
   bội của    25: 2026 // 25 =   81 → cộng thêm 81
   bội của   125: 2026 // 125 =   16 → cộng thêm 16
   bội của   625: 2026 // 625 =    3 → cộng thêm 3
→ 505 số 0 cuối
kiểm tra trên 2026!: 505


## Đáp án Bài 2

In [21]:
def answer_2_trailing_zeroes(n: int) -> int:
    return Solution().trailingZeroes(n)

assert answer_2_trailing_zeroes(3) == 0
assert answer_2_trailing_zeroes(5) == 1
assert answer_2_trailing_zeroes(25) == 6
assert answer_2_trailing_zeroes(10000) == 2499
print("Bài 2 PASS (giống Solution ở mục 7)")

Bài 2 PASS (giống Solution ở mục 7)


## Đáp án Bài 3

In [22]:
def exponent(x: int, p: int) -> int:
    e = 0
    while x % p == 0:
        x //= p
        e += 1
    return e

def answer_3_zeros_of_product(nums: list) -> int:
    twos = sum(exponent(x, 2) for x in nums)
    fives = sum(exponent(x, 5) for x in nums)
    return min(twos, fives)

assert answer_3_zeros_of_product([4, 25, 7]) == 2
assert answer_3_zeros_of_product([5, 5, 5, 2]) == 1
assert answer_3_zeros_of_product([3, 7]) == 0
from math import prod
for _ in range(2000):
    a = [random.randint(1, 200) for _ in range(random.randint(1, 6))]
    s = str(prod(a))
    assert answer_3_zeros_of_product(a) == len(s) - len(s.rstrip("0"))
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [23]:
def legendre(n: int, p: int) -> int:
    v = 0
    while n:
        n //= p
        v += n
    return v

def factorize(b: int) -> dict:
    f, p = {}, 2
    while p * p <= b:
        while b % p == 0:
            f[p] = f.get(p, 0) + 1
            b //= p
        p += 1
    if b > 1:
        f[b] = f.get(b, 0) + 1
    return f

def answer_4_trailing_zeroes_base(n: int, b: int) -> int:
    return min(legendre(n, p) // e for p, e in factorize(b).items())

def zeros_in_base(x, b):
    z = 0
    while x % b == 0:
        x //= b; z += 1
    return z

assert answer_4_trailing_zeroes_base(10, 10) == 2
assert answer_4_trailing_zeroes_base(5, 2) == 3
assert answer_4_trailing_zeroes_base(10, 12) == 4
assert answer_4_trailing_zeroes_base(1, 7) == 0
for _ in range(2000):
    n, b = random.randint(0, 60), random.randint(2, 40)
    assert answer_4_trailing_zeroes_base(n, b) == zeros_in_base(factorial(n), b), (n, b)
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [24]:
def answer_5_preimage_size(k: int) -> int:
    lo, hi = 0, 5 * k + 5
    while lo < hi:                                  # x nhỏ nhất với f(x) >= k
        mid = (lo + hi) // 2
        if Solution().trailingZeroes(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return 5 if Solution().trailingZeroes(lo) == k else 0

assert answer_5_preimage_size(0) == 5
assert answer_5_preimage_size(5) == 0
assert answer_5_preimage_size(3) == 5
assert answer_5_preimage_size(10**9) == 5
from collections import Counter
cnt = Counter(Solution().trailingZeroes(x) for x in range(0, 2000))
for k in range(0, 300):
    assert answer_5_preimage_size(k) == cnt.get(k, 0), k
print("Bài 5 PASS")
print("các k không đạt được (≤ 30):", [k for k in range(31) if answer_5_preimage_size(k) == 0])

Bài 5 PASS
các k không đạt được (≤ 30): [5, 11, 17, 23, 29, 30]


Giá trị `k = 5` bị "nhảy qua" vì `f(24) = 4` còn `f(25) = 6` — số 25 góp **hai** thừa số 5 cùng lúc.

# 14. Tổng kết

> Số 0 cuối của `n!` = số thừa số 5 (thừa số 2 luôn nhiều hơn) = `⌊n/5⌋ + ⌊n/25⌋ + ⌊n/125⌋ + …` (Legendre). Viết gọn: `while n >= 5: n //= 5; count += n`. `O(log n)`, không tính `n!`.

---

## Checklist trước khi nộp

- [ ] Đếm cả bội của 25, 125, … (không chỉ `n // 5`)?
- [ ] Không tính `n!`?
- [ ] `n = 0` → `0`?
- [ ] Dùng `n //= 5` (không nhân lũy thừa có thể tràn)?